In [3]:
import csv
import os

In [29]:
with open(os.path.normpath(os.path.join(os.getcwd(), "../", "../", "csv", "cloud_inventory.csv")), 'r') as f:
    # reader = csv.reader(f) 
    reader = csv.DictReader(f)
    print(reader)
    # next(reader)  # Skip the header row

    for row in reader:
        region = row["region"]
        uptime_days = int(row["uptime_days"])
        monthly_cost = float(row["monthly_cost"])
        row["total_cost"] = (uptime_days//12)*12
        if (region == "us-east-1" and uptime_days>=30):
            print(row)
            

{'vm_id': 'i-2eef4462', 'region': 'us-east-1', 'status': 'pending', 'uptime_days': '347', 'monthly_cost': '271.57', 'total_cost': 336}
{'vm_id': 'i-43d38b9f', 'region': 'us-east-1', 'status': 'terminated', 'uptime_days': '151', 'monthly_cost': '172.55', 'total_cost': 144}
{'vm_id': 'i-3987e4ca', 'region': 'us-east-1', 'status': 'terminated', 'uptime_days': '280', 'monthly_cost': '297.92', 'total_cost': 276}
{'vm_id': 'i-d3ca7766', 'region': 'us-east-1', 'status': 'stopped', 'uptime_days': '144', 'monthly_cost': '156.83', 'total_cost': 144}
{'vm_id': 'i-c8c4d1a6', 'region': 'us-east-1', 'status': 'stopped', 'uptime_days': '231', 'monthly_cost': '212.3', 'total_cost': 228}
{'vm_id': 'i-59711ad1', 'region': 'us-east-1', 'status': 'running', 'uptime_days': '125', 'monthly_cost': '304.99', 'total_cost': 120}
{'vm_id': 'i-185649e2', 'region': 'us-east-1', 'status': 'stopped', 'uptime_days': '175', 'monthly_cost': '55.19', 'total_cost': 168}
{'vm_id': 'i-e044014f', 'region': 'us-east-1', 'sta

In [22]:
with open('output.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    # writer.writerow(['ID', 'Status'])  # Write a single row
    
    for row in reader:
        writer.writerows(row) 

ValueError: I/O operation on closed file.

In [ ]:
import csv
import os

# 1. Properly locate the path
file_path = os.path.normpath(os.path.join(
    os.getcwd(), "../../csv/cloud_inventory.csv"))
data_to_write = []

# 2. Read and store data
with open(file_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["vm_id"])
        # Store the data so we can use it outside this block
        data_to_write.append(row)

# 3. Write data to new file
with open('output.csv', 'w', newline='') as f:
    # If using standard writer, you define columns manually
    writer = csv.writer(f)
    writer.writerow(['ID', 'Status'])  # Header

    for row in data_to_write:
        # row is a dict, so we grab specific keys or all values
        writer.writerow([row["vm_id"], "Processed"])

In [26]:
fields = ['id', 'region']
data = [{'id': 'vm-123', 'region': 'us-east'},
        {'id': 'vm-123', 'region': 'us-east'}]

with open('export.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(data)

In [19]:
import csv
from collections import defaultdict


def clean_cpu(val):
    try:
        # Strip whitespace and check if it's a "null-like" string
        if str(val).strip().upper() == "NULL" or val is None:
            return 0.0
        return float(val)
    except (ValueError, TypeError):
        return 0.0
    
region_based_dict = defaultdict(set)
cluster_based_dict = defaultdict(set)

with open("/Users/aryansingh/Documents/devops/csv/telemetry.csv","r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        vm_id = row["vm_id"]
        cluster_id = row["cluster_id"]
        region = row["region"]
        cpu_usage = clean_cpu(row["cpu_usage"])
        monthly_cost = float(row.get("monthly_cost") or 0)
        

        cluster_based_dict[cluster_id].add((vm_id,cpu_usage,monthly_cost))
        region_based_dict[region].add(cluster_id)
        
    #need to find a cluster inside which all vms have usage less than 10%
    # cluster_based_dict.items()


valid_clusters = set()
for cluster_id in cluster_based_dict.keys():
    cluster_flag = True
    for vm_id, cpu_usage, monthly_cost in cluster_based_dict[cluster_id]:
        if cpu_usage < 50:
            pass
        else:
            cluster_flag = False
            break
    if cluster_flag:
        valid_clusters.add(cluster_id)

result = set()
for valid_cluster_id in valid_clusters:
    counter = 0
    # we need to check if it spans in more than 1 region
    for region, clusters_list in region_based_dict.items():
        if(valid_cluster_id in clusters_list):
            counter+=1
        if counter >= 2:
            result.add(valid_cluster_id)
            break
    
        
    
# print(cluster_based_dict)
# print(cluster_based_dict.items())
# print(region_based_dict)
# print(region_based_dict.items())
    
print(result)    


# Writing a simple summary
with open("decommission_report.txt", "w") as f:
    f.write(f"Clusters identified: {len(result)}\n")
    for cluster in result:
        f.write(f"ID: {cluster}\n")

{'cluster-001', 'cluster-002'}


In [24]:
with open("decommission_report.txt", "r") as f:
    # reader = f.readline()
    for line in f:
        print(line)

Clusters identified: 2

ID: cluster-001

ID: cluster-002

